# Exploracion de siniestros viales en Bogota

Notebook de apoyo: consulta el modelo estrella (`data/warehouse.db`) y
explora los KPIs. Ejecuta antes `python src/run_all.py` para generar la base.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

DB = Path('..') / 'data' / 'warehouse.db'
con = sqlite3.connect(DB)

consulta = '''
SELECT t.anio, g.gravedad, COUNT(*) AS siniestros
FROM fact_siniestros f
JOIN dim_tiempo t ON t.id_tiempo = f.id_tiempo
JOIN dim_gravedad g ON g.id_gravedad = f.id_gravedad
GROUP BY t.anio, g.gravedad
ORDER BY t.anio;
'''
pd.read_sql_query(consulta, con).pivot(index='anio', columns='gravedad', values='siniestros')

In [ ]:
consulta = '''
SELECT l.nombre_localidad, COUNT(*) AS siniestros,
       ROUND(COUNT(*) * 100000.0 /
             (SELECT AVG(p.poblacion) FROM poblacion_localidad p
              WHERE p.codigo_localidad = l.codigo_localidad AND p.anio BETWEEN 2015 AND 2021), 1) AS siniestros_x100k
FROM fact_siniestros f
JOIN dim_localidad l ON l.codigo_localidad = f.id_localidad
WHERE l.codigo_localidad > 0
GROUP BY l.nombre_localidad
ORDER BY siniestros_x100k DESC;
'''
df = pd.read_sql_query(consulta, con)
df.head(10)

In [ ]:
con.close()